> layer norm 还是 rms norm，都是linear变化（shift + scaling）

- 当输入具有较大的离散度（more spread），输出斜率会变平（output slope flattens out），尤其是输入向量包含异常（outlier）值时
    - 当输入高方差时，归一化会压缩其输出值，保持在稳定、有界的范围内

### layer norm / RMSNorm

In [6]:
Image(url='./layernorm_3d_viz.png', width=1000)

- LayerNorm 是沿 feature 维（不是 batch/seq 维）归一化，所以对每个 token $x\in\mathbb{R}^d$ 单独做。
    - 对于 $d=3$，减均值后所有点落到平面 $x_1+x_2+x_3=0$ 上，再除标准差后 $\lVert x\rVert=\sqrt{d}=\sqrt3$ —— 平面 ∩ 球面 = 一个圆。

$$
\begin{split}
\mu = \frac{1}{d}\sum_{i=1}^{d} x_i,\qquad \sigma^2 = \frac{1}{d}\sum_{i=1}^{d}(x_i-\mu)^2,\qquad y_i = \frac{x_i-\mu}{\sigma}\\
\|y\|^2 = \sum_{i=1}^{d} y_i^2 = \sum_{i=1}^{d}\frac{(x_i-\mu)^2}{\sigma^2} = \frac{1}{\sigma^2}\underbrace{\sum_{i=1}^{d}(x_i-\mu)^2}_{=d\sigma^2\;(\text{方差定义直接移项})} = \frac{d\sigma^2}{\sigma^2} = d
\end{split}
$$

-----

- ① → ②　减均值 = 正交投影。$x-\mu\mathbf{1} = Px$，其中 $P = I - \frac{1}{d}\mathbf{1}\mathbf{1}^\top$ 是到 $\mathbf{1}=(1,1,1)^\top$ 正交补空间的投影矩阵（$P^2=P$，特征值为 ${0,1,1}$）。结果落在平面 $x_1+x_2+x_3=0$ 上，维度从 3 降到 2。
- ② → ③　除标准差 = 投到球面。$\lVert Px\rVert^2 = d\sigma^2$，所以归一化后 $\lVert \text{LN}(x)\rVert = \sqrt{d}\cdot\sqrt{\sigma^2/(\sigma^2+\epsilon)} \approx \sqrt{3} \approx 1.732$。平面 ∩ 球面 = 圆，自由度只剩 $d-2=1$。

-----

$\mathrm{RMS}(x)=\sqrt{\frac1d\sum x_i^2}=\lVert x\rVert/\sqrt d$，所以 $y=\sqrt d,x/\lVert x\rVert$ —— 只有球面约束，没有平面约束，因为它不减均值。